In [1]:
!pip install python-dotenv langchain-openai langchain-core


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# [1]  MessagesPlaceholder, RunnableWithMessageHistory

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
load_dotenv()

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 프롬프트: history 자리에 이전 대화가 들어감
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 오즈 내과의 상담 AI입니다. 환자가 이전에 말한 증상과 정보를 기억하고 일관되게 응대하세요."),
    MessagesPlaceholder(variable_name="history"),   # 지난 대화가 주입되는 자리
    ("human", "{question}"),
])

chain = prompt | llm                 # 이미 아는 LCEL 파이프

store = {}                           # 세션ID(환자)별 대화 저장소 (데모용 메모리)

def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="question",    # 입력 중 무엇이 환자 발화인지
    history_messages_key="history",   # 프롬프트의 어느 자리에 넣을지 (위 placeholder와 일치)
)

# session_id(환자 구분)만 넘기면 진료 대화 이력이 알아서 쌓이고 주입된다
r1 = chat.invoke(
    {"question": "어제부터 머리가 지끈거리고 뒷목이 뻐근해요."},
    config={"configurable": {"session_id": "patient_kim"}},
)
print("AI:", r1.content)

r2 = chat.invoke(
    {"question": "제가 아까 말한 증상이 뭐였죠? 혈압이랑 관련 있을까요?"},
    config={"configurable": {"session_id": "patient_kim"}},
)
print("AI:", r2.content)
# → 앞서 말한 "두통, 뒷목 뻐근함"을 기억하고 그에 이어서 답변함

C:\Users\Admin\AppData\Local\Temp\ipykernel_19872\3171301595.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\Admin\AppData\Local\Temp\ipykernel_19872\3171301595.py:23: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[sessio

AI: 머리가 지끈거리고 뒷목이 뻐근한 증상이 있으시군요. 이러한 증상은 긴장성 두통이나 스트레스, 혹은 잘못된 자세로 인한 근육 긴장 때문일 수 있습니다. 충분한 휴식을 취하고, 따뜻한 찜질을 해보시는 것도 도움이 될 수 있습니다. 증상이 지속되거나 악화된다면 병원을 방문하여 정확한 진단을 받는 것이 좋습니다. 다른 증상이나 궁금한 점이 있으시면 말씀해 주세요.
AI: 아까 말씀하신 증상은 머리가 지끈거리고 뒷목이 뻐근한 것이었습니다. 이러한 증상은 혈압과 관련이 있을 수도 있습니다. 고혈압의 경우 두통이나 목의 뻐근함을 유발할 수 있습니다. 혈압을 정기적으로 측정해보시고, 만약 혈압이 높다면 의사와 상담하여 적절한 조치를 취하는 것이 중요합니다. 추가로 궁금한 점이 있으면 언제든지 말씀해 주세요.


In [3]:
r3 = chat.invoke(
    {"question": "혈압을 재보니 150에 95가 나왔어요. 높은 편인가요?"},
    config={"configurable": {"session_id": "patient_kim"}},
)
print("AI:", r3.content)

r4 = chat.invoke(
    {"question": "그럼 제가 지금까지 말한 내용 정리해서 알려주세요."},
    config={"configurable": {"session_id": "patient_kim"}},
)
print("AI:", r4.content)
# → 두통·뒷목 뻐근함 → 고혈압 가능성 → 150/95 측정까지 누적된 맥락을 종합해 답함

AI: 네, 혈압이 150에 95라면 정상 범위보다 높은 편입니다. 일반적으로 정상 혈압은 120/80 mmHg 이하로 간주됩니다. 140/90 mmHg 이상이면 고혈압으로 진단될 수 있습니다. 고혈압은 심혈관 질환의 위험을 증가시킬 수 있으므로, 생활 습관 개선이나 약물 치료가 필요할 수 있습니다. 정확한 진단과 치료를 위해 병원을 방문하여 의사와 상담하시는 것이 좋습니다. 혈압 관리에 대해 더 궁금한 점이 있으면 말씀해 주세요.
AI: 물론입니다. 지금까지 말씀하신 내용을 정리해 드리겠습니다:

1. 어제부터 머리가 지끈거리고 뒷목이 뻐근한 증상이 있습니다.
2. 이러한 증상이 혈압과 관련이 있을지 궁금해 하셨습니다.
3. 혈압을 측정해보니 150에 95가 나왔습니다.
4. 이 혈압 수치는 정상 범위보다 높은 편이며, 고혈압으로 간주될 수 있습니다.

이 정보를 바탕으로, 고혈압 관리와 관련하여 의사와 상담하시는 것이 좋습니다. 추가로 궁금한 점이나 도움이 필요하시면 언제든지 말씀해 주세요.


In [4]:
for msg in store["patient_kim"].messages:
    print(f"[{msg.type}] {msg.content}")

[human] 어제부터 머리가 지끈거리고 뒷목이 뻐근해요.
[ai] 머리가 지끈거리고 뒷목이 뻐근한 증상이 있으시군요. 이러한 증상은 스트레스, 긴장성 두통, 또는 잘못된 자세로 인한 근육 긴장 등 여러 가지 원인으로 발생할 수 있습니다. 최근에 스트레스를 많이 받거나, 컴퓨터 앞에서 오랜 시간 앉아 있는 경우가 있었나요? 또는 다른 증상이나 건강 상태 변화가 있었는지도 알려주시면 도움이 될 것 같습니다.
[human] 제가 아까 말한 증상이 뭐였죠? 혈압이랑 관련 있을까요?
[ai] 말씀하신 증상은 머리가 지끈거리고 뒷목이 뻐근한 것이었습니다. 이러한 증상은 고혈압과 관련이 있을 수도 있습니다. 고혈압은 두통이나 목의 뻐근함을 유발할 수 있기 때문에, 혈압을 체크해보는 것이 좋겠습니다. 만약 혈압이 높다면, 의료 전문가와 상담하여 적절한 조치를 취하는 것이 중요합니다. 최근에 혈압을 측정한 적이 있나요?
[human] 혈압을 재보니 150에 95가 나왔어요. 높은 편인가요?
[ai] 네, 혈압이 150/95mmHg라면 정상 범위보다 높은 편입니다. 일반적으로 정상 혈압은 120/80mmHg 이하로 간주됩니다. 140/90mmHg 이상이면 고혈압으로 진단될 수 있습니다. 고혈압은 두통이나 목의 뻐근함과 같은 증상을 유발할 수 있으므로, 이러한 증상이 지속되거나 악화된다면 의료 전문가와 상담하는 것이 좋습니다. 생활 습관 개선이나 약물 치료가 필요할 수 있으니, 가까운 병원이나 의원을 방문해 보시길 권장합니다.
[human] 그럼 제가 지금까지 말한 내용 정리해서 알려주세요.
[ai] 물론입니다. 지금까지 말씀하신 내용을 정리해드리겠습니다:

1. 어제부터 머리가 지끈거리고 뒷목이 뻐근한 증상이 있습니다.
2. 혈압을 측정해보니 150/95mmHg로 나왔습니다.
3. 이러한 증상과 혈압 수치는 고혈압과 관련이 있을 수 있습니다.

이러한 정보를 바탕으로, 증상이 지속되거나 악화될 경우 의료 전문가와 상담하시는 것이 좋겠습니다. 추가로 궁금한 점이나 다른 증상이 

# [2] RunnablePassthrough

## [2-1] 에러
-  환자가 실제로 물은 건 "제가 먹는 약이랑 두통이 관련 있을까요?"인데, 그 원본 질문은 프롬프트에 들어가지도 않았습니다.
- question 자리까지 get_patient_history가 채워버려서, LLM은 진료기록 두 덩어리만 보고 "두통 이력이 있으니 그 얘기를 해주자"는 식으로 알아서 추측해 답을 지어낸 것입니다.
- 즉 지금 답변은 환자 질문에 답한 게 아니라, 진료기록을 보고 LLM이 임의로 화제를 정해 응답한 것입니다.
- 질문이 달랐어도, 예를 들어 "다음 진료는 언제인가요?"라고 물었어도 질문이 무시되고 있기 때문에 똑같이 두통·혈압 얘기가 출력됩니다. (예, 강아지 라고 입력해봄
- 이게 바로 RunnablePassthrough가 필요한 이유입니다. question 자리를  바꾸면 원본 질문이 그대로 프롬프트에 들어가고, LLM이 "약과 두통의 관련성"이라는 실제 물음에 초점을 맞춰 답하게 됩니다.

In [4]:
##########################
## RunnablePassthrough필요한 에러코드
###############################

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 진료기록 DB를 흉내 낸 함수 (실무에서는 환자 ID로 DB 조회)
def get_patient_history(patient_msg: str) -> str:
    return "2024-11 고혈압 진단, 혈압약(암로디핀) 복용 중. 두통 호소 이력 2회."


rag_prompt = ChatPromptTemplate.from_template(
    """다음 진료 기록을 근거로 환자 질문에 답하세요.

[진료 기록]
{context}

[환자 질문]
{question}"""
)

#  문제 코드 — question 자리에도 DB 조회 함수를 그대로 연결
#    원본 질문이 사라지고, [환자 질문] 자리에까지 진료기록이 들어감
rag_chain = (
    {
        "context":  get_patient_history,
        "question": get_patient_history,   # 여기가 문제: 원본 질문이 보존되지 않음
    }
    | rag_prompt
    | llm
    | parser
)

#print(rag_chain.invoke("제가 먹는 약이랑 두통이 관련 있을까요?"))
print(rag_chain.invoke("강아지"))

환자님, 고혈압 진단을 받으시고 현재 암로디핀을 복용 중이신 것으로 확인되었습니다. 두통을 두 번 호소하신 이력이 있으신데, 고혈압과 관련된 두통일 수 있습니다. 고혈압이 조절되지 않으면 두통이 발생할 수 있으므로, 혈압이 잘 관리되고 있는지 확인하는 것이 중요합니다. 

혹시 두통이 지속되거나 심해지면, 반드시 의료진과 상담하시기 바랍니다. 또한, 혈압약 복용 후의 부작용이나 다른 증상에 대해서도 주의 깊게 살펴보셔야 합니다. 추가적인 질문이나 걱정이 있으시면 언제든지 말씀해 주세요.


## [2-2] 올바른 답변

In [5]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context":  get_patient_history,
        "question": RunnablePassthrough(),   # 원본 질문을 그대로 보존
    }
    | rag_prompt
    | llm
    | parser
)

#print(rag_chain.invoke("제가 먹는 약이랑 두통이 관련 있을까요?"))
print(rag_chain.invoke("강아지"))

환자님, 질문하신 "강아지"에 대한 내용이 진료 기록과 관련이 없는 것 같습니다. 혹시 강아지와 관련하여 궁금한 점이나, 고혈압이나 두통과 관련된 질문이 있으신가요? 추가적인 정보를 주시면 더 정확하게 답변드릴 수 있습니다.


# [3] RunnableParallel
- 프롬프트는 입력으로 딕셔너리(mapping)를 받기를 기대하는데 문자열 "요즘 두통이 자주 있어요"를 그대로 넣었기 때문에 변수를 채우는 단계까지 가지도 못하고 TypeError: Expected mapping type ... Received <class 'str'>에서 막힙니다.
- 즉 일직선 체인은 입력값 하나를 그대로 흘려보낼 뿐, 하나의 입력을 symptoms·history 같은 여러 변수로 나눠 딕셔너리로 만들어 주지 못합니다.
- 이 "문자열 하나 → 여러 변수 딕셔너리" 변환을 담당하는 것이 바로 RunnableParallel입니다

## [3-1] 에러

In [6]:
# 에러코드임
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 진료기록 DB를 흉내 낸 함수 (실무에서는 환자 ID로 DB 조회)
def get_patient_history(patient_msg: str) -> str:
    return "2024-11 고혈압 진단, 혈압약(암로디핀) 복용 중. 두통 호소 이력 2회."


# 프롬프트는 변수 2개({history}, {symptoms})를 요구
medical_prompt = ChatPromptTemplate.from_template(
    """당신은 오즈 내과의 상담 AI입니다.
아래 환자의 과거 진료 이력을 반드시 참고하여 답변하세요.

[과거 진료 이력]
{history}

[환자의 현재 증상]
{symptoms}

위 이력을 고려해 주의가 필요한 점을 1~2문장으로 안내하세요."""
)


# ❌ 일반적인 방법 — 입력은 하나인데 변수는 두 개라서 에러 발생
medical_chain = medical_prompt | llm | parser

print(medical_chain.invoke("요즘 두통이 자주 있어요"))

TypeError: Expected mapping type as input to ChatPromptTemplate. Received <class 'str'>.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT 

## [3-2] 실행

In [7]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 진료기록 DB를 흉내 낸 함수 (실무에서는 환자 ID로 DB 조회)
def get_patient_history(patient_msg: str) -> str:
    return "2024-11 고혈압 진단, 혈압약(암로디핀) 복용 중. 두통 호소 이력 2회."


# 프롬프트는 변수 2개({history}, {symptoms})를 요구
medical_prompt = ChatPromptTemplate.from_template(
    """당신은 오즈 내과의 상담 AI입니다.
아래 환자의 과거 진료 이력을 반드시 참고하여 답변하세요.

[과거 진료 이력]
{history}

[환자의 현재 증상]
{symptoms}

위 이력을 고려해 주의가 필요한 점을 1~2문장으로 안내하세요."""
)


#  해결 — 문자열 하나를 받아 두 변수(symptoms, history) 딕셔너리로 만들어 줌
medical_parallel = RunnableParallel({
    "symptoms": lambda x: x.strip(),               # 환자 입력 가공
    "history":  lambda x: get_patient_history(x),  # DB 조회 결과
})

medical_chain = medical_parallel | medical_prompt | llm | parser

print(medical_chain.invoke("요즘 두통이 자주 있어요"))

최근 두통이 자주 발생하고 있는 점은 고혈압과 관련이 있을 수 있으므로, 혈압 관리와 두통의 원인을 정확히 파악하기 위해 반드시 진료를 받으시기 바랍니다. 또한, 현재 복용 중인 혈압약의 효과와 부작용에 대해서도 상담이 필요합니다.


# [4] RunnableBranch

## [4-1] 문제: 하나의 프롬프트로 모든 질문을 처리할 때

In [8]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 예약, 증상, 인사를 전부 한 프롬프트에 욱여넣음
mega_prompt = ChatPromptTemplate.from_template(
    """당신은 병원 챗봇입니다. 아래 질문이 예약 문의면 예약 절차를 안내하고,
증상 문의면 진료 상담을 하고, 단순 인사면 인사로 답하세요.
또한 예약이면 가능한 날짜를, 증상이면 주의사항을, 인사면 짧게...
(이런 식으로 모든 경우를 다 설명해야 해서 프롬프트가 계속 길어짐)

질문: {question}"""
)

mega_chain = mega_prompt | llm | parser

print(mega_chain.invoke("안녕하세요"))

안녕하세요! 어떻게 도와드릴까요?


## [4-2] RunnableBranch로 의도별 분기

In [ ]:
from langchain_core.runnables import RunnableBranch

# 진료기록 DB를 흉내 낸 함수
def get_patient_history(patient_msg: str) -> str:
    return "2024-11 고혈압 진단, 혈압약(암로디핀) 복용 중. 두통 호소 이력 2회."

# ① 분기될 전용 체인 3개 ─────────────────────────
reservation_prompt = ChatPromptTemplate.from_template(
    "당신은 병원 예약 담당자입니다. 다음 예약 문의에 친절히 응대하세요.\n\n문의: {question}"
)
reservation_chain = reservation_prompt | llm | parser

medical_prompt = ChatPromptTemplate.from_template(
    """오즈 내과 상담 AI입니다. 과거 이력을 참고해 답변하세요.

[과거 진료 이력]
{history}

[환자 증상/질문]
{question}"""
)
medical_chain = (
    {
        "history": lambda x: get_patient_history(x["question"]),
        "question": lambda x: x["question"],
    }
    | medical_prompt | llm | parser
)

general_prompt = ChatPromptTemplate.from_template(
    "당신은 병원 안내 AI입니다. 다음 인사에 짧고 친절하게 답하세요.\n\n{question}"
)
general_chain = general_prompt | llm | parser

# ② 의도 분류기 — intent 키를 붙여줌 ──────────────
def classify_intent(x: dict) -> dict:
    q = x["question"]
    if "예약" in q:
        intent = "예약"
    elif any(w in q for w in ["아파", "통증", "증상", "두통", "열"]):
        intent = "증상"
    else:
        intent = "일반"
    return {"question": q, "intent": intent}

# ③ RunnableBranch — 의도에 따라 분기 ─────────────
branch_router = RunnableBranch(
    (lambda x: x["intent"] == "예약", reservation_chain),  # 조건 참이면 예약 체인
    (lambda x: x["intent"] == "증상", medical_chain),      # 조건 참이면 진료 체인
    general_chain,                                         # 어디에도 안 걸리면 기본 체인
)

# ④ 분류 → 분기 연결
final_chain = classify_intent | branch_router

# 테스트
print(final_chain.invoke({"question": "내일 진료 예약 가능한가요?"}))
print("="*100)
print(final_chain.invoke({"question": "요즘 두통이 심해요"}))
print("="*100)
print(final_chain.invoke({"question": "안녕하세요"}))

## [4-3] 토큰소비량 분석

In [ ]:
from langchain_community.callbacks.manager import get_openai_callback

# mega 방식 — 인사 하나에도 모든 지침이 프롬프트에 들어감
with get_openai_callback() as cb:
    mega_chain.invoke("안녕하세요")
    print("[mega(분기 없음)]")
    print(f"  프롬프트 토큰 : {cb.prompt_tokens}")
    print(f"  응답 토큰     : {cb.completion_tokens}")
    print(f"  총 토큰       : {cb.total_tokens}")
    print(f"  비용($)       : {cb.total_cost:.6f}")

print()

# branch 방식 — 인사는 짧은 general 체인으로만 처리
with get_openai_callback() as cb:
    final_chain.invoke({"question": "안녕하세요"})
    print("[branch(분기 있음)]")
    print(f"  프롬프트 토큰 : {cb.prompt_tokens}")
    print(f"  응답 토큰     : {cb.completion_tokens}")
    print(f"  총 토큰       : {cb.total_tokens}")
    print(f"  비용($)       : {cb.total_cost:.6f}")

# [5] RunnableSequence

In [ ]:
chatbot = classify_intent | branch_router   # 이미 RunnableSequence

# ① invoke — 한 번에 실행
chatbot.invoke({"question": "요즘 두통이 심해요"})

# ② stream — 답변을 토큰 단위로 실시간 출력 (챗봇 UI에 적합)
for token in chatbot.stream({"question": "요즘 두통이 심해요"}):
    print(token, end="", flush=True)

# ③ batch — 여러 환자 메시지를 한 번에 처리
chatbot.batch([
    {"question": "내일 예약 가능한가요?"},
    {"question": "두통이 있어요"},
    {"question": "안녕하세요"},
])

# ④ ainvoke — 비동기 실행 (대규모 동시 요청 처리)
# await chatbot.ainvoke({"question": "내일 예약 되나요?"})

# [6] 실패대비

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("환자 증상에 답하세요: {question}")

# 실패 시 최대 3번까지 자동 재시도
chain = prompt | llm.with_retry(stop_after_attempt=3) | parser

print(chain.invoke({"question": "두통이 있어요"}))

# [종합] 진료기록 RAG 종합 실습 — LangChain 핵심 컴포넌트 총정리

업로드한 `진료기록_30.json`(환자 30명의 진료기록)을 이용해, 지금까지 배운
LangChain 컴포넌트를 **하나의 진료 상담 챗봇**으로 통합합니다.

| 구분 | 컴포넌트 | 이 노트북에서의 역할 |
|------|----------|----------------------|
| **4대 핵심** | `RunnableParallel` · `RunnablePassthrough` | RAG 체인의 심장 (검색 + 질문 동시 처리) |
| **4대 핵심** | `RunnableBranch` | 의도별 분기 (진료상담 / 통계 / 일반) |
| **4대 핵심** | `RunnableSequence` (`\|`) | 전체 파이프라인 컴파일 |
| 함수 끼우기 | `RunnableLambda` | 체인 중간에 내 함수 삽입 |
| 대화 기억 | `RunnableWithMessageHistory` · `MessagesPlaceholder` | 멀티턴 |
| 출력 정리 | `StrOutputParser` · `with_structured_output` | 답 정리 / 구조화 추출 |
| 안정성 | `with_retry` · `with_fallbacks` · `stream` | 실패 대비 · 스트리밍 |

> **실행 순서:** 위에서 아래로 차례대로 실행하세요. 앞 셀의 객체를 뒤 셀이 사용합니다.

### [종합1] 환경준비

`.env` 파일(이 노트북과 같은 폴더)에 아래 한 줄이 있어야 합니다:
```
OPENAI_API_KEY=sk-...
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # .env 의 OPENAI_API_KEY 로드

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 진료 상담은 일관성이 중요 → temperature=0
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()  # AIMessage → 문자열

print("환경 준비 완료:", llm.model_name)

###[종합2] 데이터 적재 — JSON을 LangChain Document로 변환
- 진료기록_30.json

벡터DB는 파이썬 dict를 그대로 받지 않습니다. 모든 문서는
**`page_content`(검색 본문)** 와 **`metadata`(부가정보)** 를 가진 `Document` 여야 합니다.


In [ ]:
import json
from langchain_core.documents import Document

with open("진료기록_30.json", "r", encoding="utf-8") as f:
    records = json.load(f)

print("총 진료기록 수:", len(records))
print(json.dumps(records[0], ensure_ascii=False, indent=2))

In [ ]:

documents = []
for r in records:
    content = (
        f"환자ID: {r['patient_id']} ({r['age']}세 {r['sex']})\n"
        f"방문일: {r['visit_date']} / 구분: {r['visit_type']}\n"
        f"증상: {', '.join(r['symptoms'])} (발현 {r['symptom_days']}일)\n"
        f"체온: {r['temperature']}도\n"
        f"진단: {r['diagnosis']}\n"
        f"처방: {r['prescription']}\n"
        f"메모: {r['note']}"
    )
    metadata = {
        "patient_id": r["patient_id"], "age": r["age"], "sex": r["sex"],
        "diagnosis": r["diagnosis"], "prescription": r["prescription"],
        "temperature": r["temperature"], "visit_type": r["visit_type"],
    }
    documents.append(Document(page_content=content, metadata=metadata))

print("변환된 Document 수:", len(documents))
print(documents[0].page_content)

### [종합3] 벡터DB 구축 — 임베딩 후 FAISS 저장
- [미리보기] 벡터DB — 자세한 내용은 다음 시간에 배웁니다.
  - 지금은 "문장을 '의미'로 검색해주는 도구를 끼운다" 정도로만 이해하고 넘어가세요.
- 왜 필요한가:
  - 단순 키워드 검색은 "머리 아파요"로 "두통" 기록을 못 찾습니다(글자가 다름).
  - 벡터DB는 문장을 '의미를 담은 숫자(임베딩)'로 바꿔 저장하므로,
  - 표현이 달라도 의미가 비슷한 기록을 찾아냅니다.

- 임베딩 모델로 문장을 벡터화하고 FAISS에 저장합니다.
  - `as_retriever`로 검색기를 만들면 이후 RAG 체인에 그대로 연결할 수 있습니다.


In [ ]:
!pip install faiss-cpu

In [ ]:

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 임베딩 모델: 문장 → 의미 벡터로 변환하는 도구
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# FAISS.from_documents: 모든 Document의 page_content를 임베딩해 벡터DB에 저장
#   (30건이라 수 초 내 완료됩니다)
vectorstore = FAISS.from_documents(documents, embeddings)

# as_retriever: 벡터DB를 '검색기'로 변환.
#   k=3 → 질문과 의미가 가까운 상위 3건을 가져오도록 설정
#   이 retriever를 뒤의 RAG 체인에서 context를 채우는 데 그대로 사용합니다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("벡터DB 구축 완료. 벡터 수:", vectorstore.index.ntotal)

In [ ]:
# 의미 검색 확인: '머리가 아파요'(질문) ↔ '두통'(기록) 매칭 여부
hits = retriever.invoke("머리가 아프고 코가 막혀요")
for i, d in enumerate(hits, 1):
    print(f"[{i}] {d.metadata['patient_id']} / {d.metadata['diagnosis']}")
    print("   ", d.page_content.split(chr(10))[2])
    print()

### [종합4] RAG 체인 — RunnableParallel + RunnablePassthrough

- JSON에서 검색한 유사 진료기록(context)과 원본 질문(question)을 묶어 답합니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

rag_prompt = ChatPromptTemplate.from_template(
    """당신은 오즈 내과의 상담 보조 AI입니다.
아래 [유사 진료기록]을 근거로 환자 질문에 답하세요.
확정 진단은 내리지 말고 가능성만 안내하며, 정확한 진단은 의료진 상담을 권유하세요.

[유사 진료기록]
{context}

[환자 질문]
{question}"""
)

rag_chain = (
    {
        "context": retriever | format_docs,   # JSON 검색 → 문자열
        "question": RunnablePassthrough(),     # 원본 질문 보존
    }
    | rag_prompt
    | llm
    | parser
)

print(rag_chain.invoke("4일째 코막힘이랑 두통이 있는데 어떤 경우일 수 있나요?"))

### [종합5] 통계 — RunnableLambda (JSON 직접 집계)

- "독감 의심 환자가 몇 명?", "가장 흔한 처방은?" 같은 **통계 질문**은
벡터 검색이 아니라 메타데이터 집계로 답해야 정확합니다.
- 일반 파이썬 함수를 RunnableLambda로 감싸면 체인 안에 그대로 넣을 수 있습니다.
- 통계 질문은 검색이 아니라 JSON 전체를 집계해야 정확합니다.


In [ ]:
from collections import Counter
from langchain_core.runnables import RunnableLambda

# JSON(records)을 직접 집계하는 함수
def diagnosis_stats(_input: str) -> str:
    dx = Counter(r["diagnosis"] for r in records)
    rx = Counter(r["prescription"] for r in records)
    avg_temp = sum(r["temperature"] for r in records) / len(records)
    lines = ["[진단 분포]"] + [f"- {k}: {v}명" for k, v in dx.most_common()]
    lines += ["", "[처방 분포]"] + [f"- {k}: {v}건" for k, v in rx.most_common()]
    lines += ["", f"[평균 체온] {avg_temp:.1f}도 (전체 {len(records)}건)"]
    return "\n".join(lines)

stats_prompt = ChatPromptTemplate.from_template(
    """아래 통계를 바탕으로 질문에 간결히 답하세요.

[통계]
{stats}

[질문]
{question}"""
)

stats_chain = (
    {
        "stats": RunnableLambda(diagnosis_stats),  # JSON 집계 함수 삽입
        "question": RunnablePassthrough(),
    }
    | stats_prompt
    | llm
    | parser
)

print(stats_chain.invoke("진단 중에 제일 많은 게 뭐고, 처방은 뭐가 흔해?"))

In [ ]:
###############################
## 판다스로 llm통계확인
###############################
import pandas as pd
import json

# JSON 로드 후 데이터프레임으로 변환
with open("진료기록_30.json", "r", encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
print("전체 건수:", len(df))

# ① 진단 분포
print("\n[진단 분포]")
dx_counts = df["diagnosis"].value_counts()
print(dx_counts)

# ② 처방 분포
print("\n[처방 분포]")
rx_counts = df["prescription"].value_counts()
print(rx_counts)

# ③ LLM 답변 검증
print("\n[검증]")
top_dx, top_dx_n = dx_counts.index[0], dx_counts.iloc[0]
top_rx, top_rx_n = rx_counts.index[0], rx_counts.iloc[0]
print(f"가장 많은 진단 : {top_dx} ({top_dx_n}명)")
print(f"가장 흔한 처방 : {top_rx} ({top_rx_n}건)")

print("\nLLM 답변과 일치 여부:")
print("  진단 = 급성 기관지염 15명 ?", (top_dx == "급성 기관지염") and (top_dx_n == 15))
print("  처방 = 해열제 8건 ?", (top_rx == "해열제") and (top_rx_n == 8))

### [종합6] 구조화 출력 — with_structured_output (JSON 검색 결과 기반 분류)

- 환자 발화만으로 추정하지 않고, JSON에서 검색한 유사 사례를 근거로 분류합니다.
  - 앞선 구조화 출력 예제는 환자 발화만 보고 LLM이 자기 일반 지식으로 추정 진단을 내렸습니다. 그러면 우리 병원의 실제 진료기록과 무관한, 교과서적 답변이 나옵니다. 이번에는 그 추정의 근거를 JSON 30건에서 가져오도록 바꿉니다.

- 데이터가 흐르는 순서는 이렇습니다.
  - 먼저 환자 발화(예: "39도 고열에 오한이 일주일째예요")가 retriever로 들어가, FAISS 벡터DB에서 의미가 가까운 진료기록 3건을 찾아옵니다.
  - 이 3건은 우리 JSON 안에 실제로 존재하는 환자 사례입니다. format_docs가 그 사례들을 하나의 문자열로 합쳐 프롬프트의 {context} 자리에 넣습니다.
  - 즉 LLM은 빈손으로 추정하는 게 아니라, "이 병원에 이런 비슷한 환자들이 있었고 그들은 이런 진단을 받았다"는 실제 데이터를 보고 판단합니다.
  - 그 결과로 similar_diagnoses 필드에는 검색된 JSON 기록들의 실제 진단명이 담깁니다. \
  - 예를 들어 고열·오한 발화로 검색하면 데이터에 있는 "독감 의심", "급성 기관지염" 같은 실제 사례 진단이 추출되고,
  - likely_diagnosis는 그 사례들을 근거로 한 추정이 됩니다.
  - 발화에서 뽑은 증상(main_symptoms)과 데이터에서 찾은 진단(similar_diagnoses)이 한 결과 객체 안에 함께 들어가는 구조입니다.

- with_structured_output의 역할
  - 검색 결과를 근거로 LLM이 판단한 내용을, 문자열이 아니라 Pydantic 객체(JSON 구조) 로 받아냅니다.
  - result.likely_diagnosis, result.urgency처럼 필드로 바로 접근할 수 있어, 이 결과를 다음 단계(예: 응급도가 '높음'이면 별도 안내 분기)에서 코드로 처리하기 쉽습니다. - 검색으로 가져온 비정형 진료기록을, 정형화된 분류 결과로 변환하는 단계라고 보면 됩니다.

- 정리
  - 이 파트의 핵심은 "LLM의 추측"을 "JSON 데이터에 근거한 추정"으로 바꾼 것입니다.
  - retriever가 JSON에서 실제 유사 사례를 찾고(검색), with_structured_output이 그 근거를 깔끔한 구조로 정리합니다(구조화).
  -  검색과 구조화가 결합되어, 우리 진료기록에 뿌리를 둔 분류 결과가 만들어집니다.


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Triage(BaseModel):
    main_symptoms: List[str] = Field(description="환자 발화에서 추출한 핵심 증상")
    similar_diagnoses: List[str] = Field(description="유사 진료기록에서 나타난 진단명들")
    likely_diagnosis: str = Field(description="유사 사례를 근거로 한 가장 가능성 있는 추정 진단")
    urgency: str = Field(description="응급도: 낮음/보통/높음 중 하나")
    advice: str = Field(description="환자에게 줄 한 줄 안내")

triage_llm = llm.with_structured_output(Triage)

triage_prompt = ChatPromptTemplate.from_template(
    """아래 [유사 진료기록]을 참고하여 환자 발화를 분류하세요.
확정이 아닌 '추정'임을 전제로 합니다.

[유사 진료기록]
{context}

[환자 발화]
{question}"""
)

# 검색(context) + 원본(question) → 구조화 LLM
triage_chain = (
    {
        "context": retriever | format_docs,   # JSON 검색을 분류 근거로 사용
        "question": RunnablePassthrough(),
    }
    | triage_prompt
    | triage_llm
)

result = triage_chain.invoke("39도 고열에 오한이 일주일째예요. 너무 힘들어요.")
print("핵심증상:", result.main_symptoms)
print("유사 진단들:", result.similar_diagnoses)
print("추정진단:", result.likely_diagnosis)
print("응급도:", result.urgency)
print("안내:", result.advice)

In [ ]:
###################################
## 검증
# 검색이 실제로 가져온 3건은 전부 상기도 감염이었습니다(P005, P030, P003).
# 그런데 LLM은 ['상기도 감염', '독감', '폐렴']이라고 답했습니다.
# 즉 검색 결과에 있던 건 "상기도 감염" 하나뿐이고, "독감"과 "폐렴"은 LLM이 자기 지식으로 지어낸 것입니다.
# 특히 폐렴은 우리 JSON 30건 어디에도 없는 진단명입니다.
####################################3

# 검증: retriever가 실제로 가져온 진료기록이 무엇인지 직접 확인
query = "39도 고열에 오한이 일주일째예요. 너무 힘들어요."

hits = retriever.invoke(query)

print("=== 검색이 실제로 가져온 기록 3건 ===")
for i, d in enumerate(hits, 1):
    print(f"[{i}] {d.metadata['patient_id']} | 진단: {d.metadata['diagnosis']} | 체온: {d.metadata['temperature']}도")

print("\n=== LLM이 'similar_diagnoses'로 답한 것 ===")
print(result.similar_diagnoses)

# 데이터에 실제로 존재하는 진단명 목록과 대조
valid_dx = set(r["diagnosis"] for r in records)
print("\n=== JSON에 실제 존재하는 진단명 ===")
print(valid_dx)

print("\n=== LLM 답변 중 데이터에 없는(지어낸) 진단 ===")
made_up = [d for d in result.similar_diagnoses if d not in valid_dx]
print(made_up if made_up else "없음 (모두 실제 데이터 기반)")

In [ ]:
#############################
# 프롬프트 제약
# 검색된 기록에 있는 진단명만 쓰도록 강하게 제약
##################################
triage_prompt = ChatPromptTemplate.from_template(
    """아래 [유사 진료기록]에 실제로 등장한 진단명만 사용하세요.
기록에 없는 진단명(독감, 폐렴 등)은 절대 추가하지 마세요.
similar_diagnoses에는 아래 기록에 나온 진단명만, 중복 없이 넣으세요.

[유사 진료기록]
{context}

[환자 발화]
{question}"""
)

triage_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | triage_prompt
    | triage_llm
)

result = triage_chain.invoke("39도 고열에 오한이 일주일째예요. 너무 힘들어요.")
print("유사 진단들:", result.similar_diagnoses)

# 다시 검증
made_up = [d for d in result.similar_diagnoses if d not in valid_dx]
print("지어낸 진단:", made_up if made_up else "없음 (모두 실제 데이터 기반)")

## [종합7] RunnableBranch — 의도별 분기 라우팅

질문을 **통계 / 진료상담 / 일반** 으로 나눠, 각기 다른 전용 체인으로 보냅니다.
모든 질문을 한 체인으로 처리하면 토큰 낭비·정확도 저하가 생기므로 분기합니다.
- 지금까지 만든 체인들은 각자 JSON을 다르게 활용합니다. RAG 체인(4장)은 벡터DB로 진료기록을 검색하고, 통계 체인(5장)은 JSON 전체를 집계합니다.
- 문제는 사용자 질문이 이 중 어느 쪽에 해당하는지 미리 알 수 없다는 점입니다.
- 두통이 5일째예요는 검색이 필요하고, 독감 의심이 몇 명이야는 집계가 필요하며, 안녕하세요는 둘 다 필요 없습니다.
- RunnableBranch는 질문의 의도를 먼저 판별한 뒤, 그에 맞는 전용 체인으로 경로를 나눠 보냅니다.
- 진료 상담이면 JSON을 검색하는 rag_chain으로, 통계면 JSON을 집계하는 stats_chain으로, 일반 인사면 검색 없는 general_chain으로 라우팅합니다.
- 즉 분기 자체가 JSON을 쓰는 것은 아니지만, 각 갈래가 앞서 만든 JSON 기반 체인으로 연결되어 질문 종류에 맞는 방식으로 데이터를 활용하게 만듭니다.


In [ ]:
from langchain_core.runnables import RunnableBranch

# 일반 인사용 짧은 체인 (검색 불필요)
general_chain = (
    ChatPromptTemplate.from_template(
        "당신은 오즈 내과 안내 AI입니다. 다음에 짧고 친절히 답하세요.\n\n{question}"
    )
    | llm | parser
)

# 의도 분류 키워드
STAT_WORDS = ["몇", "통계", "분포", "가장", "제일", "평균", "비율"]
MED_WORDS  = ["아파", "아픈", "증상", "두통", "기침", "열", "콧물", "오한",
              "인후", "근육통", "가래", "피로", "코막힘", "진단", "처방", "재채기"]

# 질문 → {question, intent} 로 의도 판별
def classify(x):
    q = x if isinstance(x, str) else x["question"]
    if any(w in q for w in STAT_WORDS):
        intent = "stats"
    elif any(w in q for w in MED_WORDS):
        intent = "medical"
    else:
        intent = "general"
    return {"question": q, "intent": intent}

# 의도에 따라 JSON 기반 체인으로 분기
branch = RunnableBranch(
    (lambda x: x["intent"] == "stats",   lambda x: stats_chain.invoke(x["question"])),  # JSON 집계
    (lambda x: x["intent"] == "medical", lambda x: rag_chain.invoke(x["question"])),    # JSON 검색
    lambda x: general_chain.invoke({"question": x["question"]}),                        # 일반(검색 없음)
)

# 분류 → 분기. | 가 곧 RunnableSequence
router_chain = RunnableLambda(classify) | branch

print("● 통계:", router_chain.invoke("독감 의심이 몇 명이야?")[:80], "...")
print("● 상담:", router_chain.invoke("두통이랑 인후통이 5일째예요")[:80], "...")
print("● 일반:", router_chain.invoke("안녕하세요"))

In [ ]:
# ─────────────────────────────────────────────────────────────
# 분류기 검증셋 — 쉬운 케이스 + 함정 케이스를 의도적으로 섞음
#   각 케이스의 '의도'를 주석으로 명시
# ─────────────────────────────────────────────────────────────
testset = [
    # --- 쉬운 케이스: 키워드가 명확해서 잘 맞아야 함 ---
    ("독감 의심 환자가 몇 명이야?", "stats",   "'몇' → 집계 질문, 명확"),
    ("가장 많이 처방된 약은?",      "stats",   "'가장' → 집계 질문, 명확"),
    ("두통이 5일째 계속돼요",       "medical", "'두통' → 증상 상담, 명확"),
    ("어제부터 기침이 심해요",      "medical", "'기침' → 증상 상담, 명확"),
    ("안녕하세요",                 "general", "키워드 없음, 단순 인사"),

    # --- 함정 케이스: 키워드 매칭이 '의도'를 못 읽어 틀리는 지점 ---
    ("처방전 재발급 받으려면 어떻게 하나요?", "general",
     "행정 문의인데 '처방' 글자 때문에 medical로 오분류됨"),
    ("열심히 운동하면 건강해질까요?",        "general",
     "잡담인데 '열' 글자 때문에 medical로 오분류됨"),
    ("진료 예약을 변경하고 싶어요",          "general",
     "예약 문의인데 분류 카테고리에 '예약'이 없어 처리 불가"),
    ("최근에 감기 환자가 늘었나요?",         "stats",
     "집계 질문인데 '몇/가장' 같은 키워드가 없어 medical/general로 샘"),
    ("요즘 몸이 영 안 좋네요",               "medical",
     "증상 호소인데 구체 증상 단어가 없어 general로 샘"),
]

correct = 0
wrong = []
for q, expected, intent_note in testset:
    got = classify(q)["intent"]
    ok = (got == expected)
    correct += ok
    mark = "✓" if ok else "✗"
    print(f"{mark} '{q}'")
    print(f"     결과={got} / 정답={expected}  ← {intent_note}")
    if not ok:
        wrong.append((q, got, expected))

print(f"\n정확도: {correct}/{len(testset)} = {correct/len(testset)*100:.0f}%")
if wrong:
    print("\n[키워드 분류기가 의도를 놓친 케이스]")
    for q, got, expected in wrong:
        print(f"  '{q}' → {got} (정답: {expected})")

## [종합8] 멀티턴 — RunnableWithMessageHistory + MessagesPlaceholder

환자가 앞서 말한 증상을 챗봇이 기억하게 합니다.
`session_id`별로 대화가 분리되어 저장됩니다.

> ⚠️ 주의: `store`는 메모리라 커널 재시작 시 사라집니다. `session_id`는
> 반드시 인증된 환자 식별자와 1:1로 묶어야 개인정보가 섞이지 않습니다.

- 이 파트가 JSON을 활용하는 방식지금까지의 체인은 모두 단발성이었습니다.
- 한 번 묻고 한 번 답하면 끝이라, 환자가 "아까 그 증상 말인데요"라고 이어 물어도 앞 대화를 기억하지 못합니다.
- 실제 진료 상담은 여러 번 주고받으며 맥락이 쌓이므로, 대화 기억이 필요합니다.
- 이 파트는 두 가지를 결합합니다. 하나는 RunnableWithMessageHistory로 이전 대화를 기억하는 멀티턴이고, 다른 하나는 진료 질문이 들어오면 JSON에서 유사 기록을 검색해 근거로 삼는 RAG입니다.
- 즉 환자가 "두통이 있어요"라고 하면 JSON을 검색해 답하고, 그 대화를 기억한 채 다음 질문("비슷한 환자들은 무슨 진단을 받았나요?")도 검색 결과와 이전 맥락을 함께 보고 답합니다. 대화 기억과 데이터 검색이 한 체인 안에서 동시에 작동합니다.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

chat_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 오즈 내과 상담 보조 AI입니다. 아래 [유사 진료기록]과 이전 대화를 함께 참고해 "
     "일관되게 응대하되, 기록에 없는 내용은 지어내지 말고, 확정 진단 대신 의료진 상담을 권유하세요.\n\n"
     "[유사 진료기록]\n{context}"),
    MessagesPlaceholder(variable_name="history"),   # 지난 대화가 들어갈 자리
    ("human", "{question}"),
])

# 입력 dict의 question으로 JSON을 검색해 context를 채우고, 원본 질문·이력은 그대로 통과
chat_core = (
    {
        "context": (lambda x: x["question"]) | retriever | format_docs,  # JSON 검색
        "question": (lambda x: x["question"]),                            # 원본 질문 보존
        "history": (lambda x: x.get("history", [])),                      # 이력 통과
    }
    | chat_prompt
    | llm
    | parser
)

store = {}  # 세션ID별 대화 저장소 (데모용 메모리)
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat = RunnableWithMessageHistory(
    chat_core, get_history,
    input_messages_key="question",     # {question} 과 일치해야 함
    history_messages_key="history",    # MessagesPlaceholder 와 일치해야 함
)

cfg = {"configurable": {"session_id": "patientA"}}
print("Q1:", chat.invoke({"question": "어제부터 두통이랑 오한이 있어요."}, config=cfg))
print()
print("Q2:", chat.invoke({"question": "비슷한 환자들은 보통 무슨 진단을 받았나요?"}, config=cfg))

In [ ]:
####################3
## 검증용

# Q2가 실제로 검색한 기록 확인 — Q2 질문 그대로 retriever에 넣어봄
q2 = "비슷한 환자들은 보통 무슨 진단을 받았나요?"
hits = retriever.invoke(q2)

print("=== Q2 질문으로 검색된 기록 ===")
for i, d in enumerate(hits, 1):
    print(f"[{i}] {d.metadata['patient_id']} | 증상포함: {d.page_content.split(chr(10))[2]}")
    print(f"     진단: {d.metadata['diagnosis']}")

# LLM이 Q2에서 언급한 사례가 데이터에 실제 있는지 대조
print("\n=== LLM이 언급한 증상조합이 JSON에 실제 존재하나? ===")
for r in records:
    syms = set(r["symptoms"])
    if {"두통", "인후통", "재채기"}.issubset(syms):
        print(f"  두통+인후통+재채기 → {r['patient_id']}: {r['diagnosis']} ✓ 실제 존재")
    if {"두통", "코막힘", "근육통"}.issubset(syms):
        print(f"  두통+코막힘+근육통 → {r['patient_id']}: {r['diagnosis']} ✓ 실제 존재")

### 위의검증으로 들어난것
- 첫째, 이번에는 환각이 없었습니다. Q2 답변이 든 두 사례(P001 급성 비인두염, P016 급성 기관지염)가 실제 검색 결과 및 데이터와 정확히 일치했습니다.
  - 6번과 달리 LLM이 검색된 기록에 충실했습니다.

- 둘째, 답변만으로는 보이지 않던 문제가 잡혔습니다.
  - 같은 증상 조합인 두통·코막힘·근육통이 데이터에서는 P003 상기도 감염과 P016 급성 기관지염으로 진단이 갈립니다.
  - 그런데 LLM은 P016만 인용해 급성 기관지염이라고 단정했습니다. 환각은 아니지만, 여러 사례 중 일부만 골라 말한 선택적 인용입니다.
  - 같은 증상도 진단이 갈릴 수 있다는 중요한 정보가 답변에서 누락된 것입니다.

- 셋째, 멀티턴 검색의 잠재적 약점도 확인했습니다. Q2 문장에는 증상 단어가 없는데도 이번에는 검색이 두통 관련 기록을 잘 가져왔습니다. 다만 이는 보장된 동작이 아니라, 질문에 따라 대화 맥락과 무관한 기록을 가져올 수도 있습니다.

- 교훈
  - LLM이 검색 결과에 충실했는지는 매번 데이터와 대조해야만 알 수 있습니다. 그리고 환각이 없어도 LLM은 데이터의 일부만 선택해 말할 수 있으므로, 전체 분포가 중요한 질문은 검색 답변을 그대로 믿지 말고 집계(5장 방식)로 보완해야 합니다. RAG 검증이 잡아내는 문제는 환각(6번)에서 선택적 인용(8번)으로, 점점 미묘한 층위로 깊어집니다.

### [종합8-2] 환각 개선 / CoT + 퓨샷을 적용한 멀티턴 RAG 전체 코드


In [ ]:
# ─────────────────────────────────────────────────────────────
# 8-B. CoT + 퓨샷으로 '선택적 인용' 잡기
#   문제: LLM이 검색된 여러 사례 중 일부만 골라 말함(같은 증상의 다른 진단 누락)
#   해결: ① 퓨샷으로 "진단이 갈리면 모두 제시"하는 답변 형식을 학습
#         ② CoT로 "먼저 기록을 전부 나열 → 갈리는지 확인 → 답변" 순서를 강제
# ─────────────────────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 퓨샷 예시: 진단이 갈릴 때 어떻게 답해야 하는지 모범 답변 1개
fewshot_example = """[좋은 답변 예시]
질문: 코막힘 환자는 보통 무슨 진단을 받나요?
검색된 기록: P003(코막힘·근육통·두통 → 상기도 감염), P016(코막힘·두통·근육통 → 급성 기관지염)
답변: 검색된 기록상 코막힘 환자는 진단이 갈립니다. 상기도 감염(P003)과 급성 기관지염(P016)
두 사례가 있어, 동일한 증상이라도 진단이 다를 수 있습니다. 정확한 진단은 의료진 상담이 필요합니다."""

# 시스템 프롬프트: 제약(환각 방지) + CoT(단계 강제) + 퓨샷(형식 학습)
cot_system = (
    "당신은 오즈 내과 상담 보조 AI입니다. 아래 [유사 진료기록]에 등장한 내용만 근거로 답하고, "
    "기록에 없는 진단명은 절대 만들지 마세요.\n\n"
    "반드시 다음 순서로 사고하여 답하세요.\n"
    "1단계: 검색된 기록을 환자ID·증상·진단까지 빠짐없이 확인한다.\n"
    "2단계: 같은(또는 비슷한) 증상인데 진단이 다른 경우가 있는지 점검한다.\n"
    "3단계: 종합해 답하되, 진단이 갈리면 일부만 고르지 말고 갈린다고 명시한다.\n"
    "확정 진단 대신 의료진 상담을 권유하세요.\n\n"
    f"{fewshot_example}\n\n"
    "[유사 진료기록]\n{context}"
)

cot_prompt = ChatPromptTemplate.from_messages([
    ("system", cot_system),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

# 검색(context) + 원본질문(question) + 이력(history) 결합
cot_chat_core = (
    {
        "context": (lambda x: x["question"]) | retriever | format_docs,  # JSON 검색
        "question": (lambda x: x["question"]),                            # 원본 보존
        "history": (lambda x: x.get("history", [])),                      # 이력 통과
    }
    | cot_prompt
    | llm
    | parser
)

# 멀티턴 래핑 (앞 8장과 별도 저장소 사용)
cot_store = {}
def get_cot_history(session_id: str):
    if session_id not in cot_store:
        cot_store[session_id] = ChatMessageHistory()
    return cot_store[session_id]

cot_chat = RunnableWithMessageHistory(
    cot_chat_core, get_cot_history,
    input_messages_key="question",
    history_messages_key="history",
)

# 같은 대화 흐름으로 실행
cfg = {"configurable": {"session_id": "patientB"}}
print("Q1:", cot_chat.invoke({"question": "어제부터 두통이랑 오한이 있어요."}, config=cfg))
print()
ans2 = cot_chat.invoke({"question": "비슷한 환자들은 보통 무슨 진단을 받았나요?"}, config=cfg)
print("Q2:", ans2)

#### [위의 내용 정리] 선택적 인용은 프롬프트로 잡았고, 남은 검색 품질 문제(관련 낮은 기록 혼입)는 다음 단계인 실전 RAG에서 다룹니다. 거기서 검색 결과를 거르거나(재순위화), 관련도 점수로 필터링하거나, 대화 맥락을 반영해 검색 질의를 다시 쓰는(query rewriting) 기법으로 해결합니다.

## [종합9] 안정성 — with_retry · with_fallbacks · stream
지금까지 만든 RAG 체인은 정상 상황에서만 동작합니다. 하지만 실제 서비스로 내보내면 네트워크 순간 끊김, API 과부하, 모델 장애 같은 일이 일어납니다. 이 파트는 4장에서 만든 JSON 기반 RAG 체인(retriever로 진료기록을 검색하는 체인)에 그대로 안정성 장치를 입힙니다. 검색 로직은 동일하게 유지한 채, 그 위에 재시도·대체·스트리밍을 덧씌우는 것입니다.세 가지 도구의 역할은 다음과 같습니다. with_retry는 일시적 오류가 나면 자동으로 다시 시도합니다. with_fallbacks는 재시도해도 안 되거나 주력 모델에 장애가 나면 대체 모델로 우회합니다. stream은 답변을 토큰 단위로 실시간 출력해 사용자 체감 속도를 높입니


실제 서비스로 내보내기 위한 마지막 장치입니다.
- with_retry : 일시적 오류(네트워크·과부하) 자동 재시도
- with_fallbacks : 주력 모델 장애 시 대체 모델로 우회
- stream : 답변을 토큰 단위로 실시간 출력

In [ ]:
# 주력 모델 + 재시도 + 대체 모델을 결합한 견고한 LLM
backup_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

robust_llm = (
    llm.with_retry(stop_after_attempt=2)      # 먼저 2회까지 재시도
       .with_fallbacks([backup_llm])          # 그래도 실패하면 대체 모델
)

robust_rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | robust_llm
    | parser
)

print(robust_rag.invoke("기침이 오래가는 환자는 보통 어떤 진단을 받았나요?"))

In [ ]:
# stream: 생성되는 즉시 토큰을 흘려보내 실시간 출력 (ChatGPT 처럼)
print("스트리밍 출력 ↓")
for chunk in robust_rag.stream("열이 39도까지 오르면 어떻게 해야 하나요?"):
    print(chunk, end="", flush=True)
print()